<a href="https://colab.research.google.com/github/JeremiahSents/Knowledge-Engineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 3: Searching Agent — Self-Driving Trailer

Find the cheapest route from Nairobi to Kisumu using Uniform Cost Search (path cost = distance in km).

In [101]:
import heapq
#This dictionary represents the graph we made on the routes to Kisumu. The key is the town and the values represent all
#other towns that it can go to
graph = {
    "Nairobi": {"Thika": 45 , "Narok": 142 , "Naivasha": 90},
    "Thika": {"Nyeri": 105},
    "Nyeri": {"Nanyuki": 60, "Nyahururu": 100},
    "Nanyuki": {},
    "Nyahururu": {"Nakuru": 70},
    "Naivasha": {"Nakuru": 70},
    "Nakuru": {"Kericho": 107, "Eldoret": 156},
    "Narok": {"Kisii": 170},
    "Kericho": {"Kisii": 100, "Kisumu": 84},
    "Eldoret": {"Kakamega": 115, "Webuye": 60},
    "Webuye": {"Kakamega": 40},
    "Kisii": {"Kisumu": 112},
    "Kakamega":{"Kisumu": 52},
}

In [102]:
#this is where we will store the candidate towns and decide which path to take(the cheapest)
priority_queue = []
#this ensures that we select the cheapest path in the heap
heapq.heappush(priority_queue, (0, "Nairobi", ["Nairobi"]))
#this is a dictionary that stores the cheapest path to a specific town from the start town(Nairobi)
best_cost = {
    "Nairobi": 0,
}

#This loops through all the available paths to Kisumu and picks the cheapest one
while priority_queue:
    # 1. Pop the cheapest entry
    current_cost,current_town,current_path = (heapq.heappop(priority_queue))
    # 2. Skip outdated entries
    if current_cost > best_cost[current_town]:
      continue
    # 3. Check if current town is Kisumu
    if current_town == "Kisumu":
      print(f"Minimum cost: {best_cost[current_town]}")
      print(f"Path: {current_path}")
      break
    # 4. Explore neighbouring towns
    for town, cost in graph[current_town].items():
      # 5. Calculate new costs
      new_cost = cost + current_cost
      if town not in best_cost or new_cost < best_cost[town]:
        # 6. Update best_cost and priority_queue
        best_cost[town] = new_cost
        new_path = current_path + [town]
        heapq.heappush(priority_queue, (new_cost,town,new_path))

Minimum cost: 351
Path: ['Nairobi', 'Naivasha', 'Nakuru', 'Kericho', 'Kisumu']


# Part 4: Knowledge Engineering — Self-Driving Trailer

Builds on Part 3: the trailer's knowledge base is Kenya's highway network.

**Step 1: Identify the Task**

Give the self-driving trailer the knowledge it needs to plan a trip from Nairobi to Kisumu.

* **Input:** the trailer's current town
* **Output:** roads it can take next, their distances, and whether Kisumu can be reached
* **Goal:** only plan routes on roads that exist

**Step 2: Assemble Relevant Knowledge**

* Towns in the Nairobi, Central Highlands, Rift Valley and Lake Victoria Basin regions
* Direct highway links between towns (one-way, as in the Part 3 directed graph)
* Distance of each link in km

**Source:** Google Maps distances from Practical 3, entered by hand. No APIs or web scraping were used.

**Step 3: Define the Vocabulary**

* **Predicates:** `Town(t)`, `Road(a, b)`, `Distance(a, b, km)`, `Reachable(a, b)`
* **Constants:** Nairobi, Thika, Nyeri, Nanyuki, Nyahururu, Naivasha, Nakuru, Narok, Kericho, Eldoret, Webuye, Kisii, Kakamega, Kisumu

**Step 4: Encode General Knowledge about the Domain** `[3 marks]`

`roads` stores every `Road(a, b)` with its `Distance(a, b, km)`; the functions below encode each predicate.

In [ ]:
# Road(a, b) and Distance(a, b, km): town -> {next town: km}
roads = {
    "Nairobi":   {"Thika": 45, "Narok": 142, "Naivasha": 90},
    "Thika":     {"Nyeri": 105},
    "Nyeri":     {"Nanyuki": 60, "Nyahururu": 100},
    "Nanyuki":   {},
    "Nyahururu": {"Nakuru": 70},
    "Naivasha":  {"Nakuru": 70},
    "Nakuru":    {"Kericho": 107, "Eldoret": 156},
    "Narok":     {"Kisii": 170},
    "Kericho":   {"Kisii": 100, "Kisumu": 84},
    "Eldoret":   {"Kakamega": 115, "Webuye": 60},
    "Webuye":    {"Kakamega": 40},
    "Kisii":     {"Kisumu": 112},
    "Kakamega":  {"Kisumu": 52},
    "Kisumu":    {},
}

def Town(t):
    return t in roads

def Road(a, b):
    return Town(a) and b in roads[a]

def Distance(a, b):
    return roads[a][b] if Road(a, b) else None

def Reachable(a, b):
    """True if the trailer can drive from a to b along known roads."""
    if not (Town(a) and Town(b)):
        return False
    visited, stack = set(), [a]
    while stack:
        town = stack.pop()
        if town == b:
            return True
        if town not in visited:
            visited.add(town)
            stack.extend(roads[town])
    return False

print("Towns in KB:", len(roads))
print("Road(Nakuru, Kericho):", Road("Nakuru", "Kericho"), "|", Distance("Nakuru", "Kericho"), "km")

**Step 5: Add New Facts to the Knowledge Base** `[4 marks]`

**Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers** `[3 marks]`

**Step 8: Querying the knowledge base**

**Step 9: Debugging the Knowledge Base** `[2 marks]`

**Step 10: Add final reflection here (challenges or recommended improvements)** `[3 marks]`